# Scriven problem Description

Testcase used in the following work:
L. Wei, G. Bois, V. Pandeya, V. S. Nikolayev. (2025). _Multiscale simulations on the influence of contact line evaporation during nucleate boiling_. Applied Thermal Engineering. 

## Objective
This 2D test case (with two axis of symetry) is widely used for the verification of phase-change models, which involves phase change driven by temperature gradients.

## Remark
In the Scriven problem, a temperature gradient within the **Liquid** region induces evaporation, which in turn drives the interface velocity. The governing equations are solved on the **Liquid side only**.

## Summary of Initial and Boundary Conditions
- **Boundary Conditions (BCs):**
  - **Dynamics**: 
    - Outlet condition on the top and right sides.
    - Symmetry on the left and bottom sides.
  - **Thermal**: 
    - Adiabatic conditions on on the left and bottom sides.
    - Fixed temperature $ T_{\infty}=T_{\text{sat}} +\Delta T_{\infty} $ at the top and right sides.
  - **Phase**: All boundaries set to symmetry.

- **Initial Conditions (ICs):**
  - **Dynamics**: Initial velocity is zero throughout the domain.
  - **Thermal**: The analytical temperature solution is imposed within the liquid, ranging from $ T_{sat} $ at $ r=R_0 $ to $  T_{\infty} $ at $ r=R_L $.
  - **Phase**: Initial radius of vapor, $ r=R_0 $.

![ICs and BCs for Stefan Problem](src/figs/scriven.png)

## Case Specifications
- **Domain Length**: $ L = 4R_0 = 4 \, \text{mm} $
- **Initial Interface Position**: $ R_0 = 1 \, \text{mm} $ at $ t = 0 \, \text{s} $
- **Material Properties**: Properties for water under atmospheric conditions:

|               | $\rho$ $kg/m^3$| $\mu$ $Pa\cdot s$        | $\lambda$ $W/(m\cdot K)$ | $C_p$ $J/(kg\cdot K)$   |
|---------------|----------------|--------------------------|--------------------------|-------------------------|
| **Liquid**    | 958.37         | $2.8 \times 10^{-4}$     | 0.679                    | $4.21 \times 10^3$      |
| **Vapor**     | 0.597          | $1.227 \times 10^{-5}$   | 0.025                    | $2.077 \times 10^3$     |
|               |                |                          |                          |                         |
| $\sigma = 0.0589$ N/m, $\mathcal{L} = 2.256 \times 10^6$ J/kg                                                  | 

---

## Analytical Solution

The analytical solution provides the theoretical interface position $ R(t) $ and the temperature distribution $ \Delta T(x, t) $ within the vapor.

$$
R(t) = 2 \beta \sqrt{\alpha_l t}
$$

$$
\Delta T_l(x, t) = \Delta T_\infty-\frac{2\beta^2}{Ja}\Delta T_\infty\int_a^1\exp\left\{-\beta^2\left[(1-y)^{-2}-2\epsilon y-1\right]\right\}\mathrm{d}y 
$$




where:
$\alpha_l$ is the thermal diffusivity of the liquid, defined by $$\alpha_l=\frac{\lambda_l}{\rho_l cp_{l}} $$; 

$cp_v$ is the specific heat capacity of the vapor (J/kg·K); $\operatorname{erf}(\beta)$ is the error function; $\beta$ is determined by solving the transcendental equation:

$$
Ja = 2\beta^2 \int_{0}^{1} \exp \left\{ -\beta^2 \left[ (1-y)^{-2} - 2\epsilon y - 1 \right] \right\} \, dy = \frac{\rho_l C_{pl} \Delta T_\infty}{\rho_v [\mathcal{L}+(C_{pl}-C_{pv})\Delta T_\infty]}
$$


## Moduels for plot,  and funcitons for analytical solutions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
from math import sqrt, pi
Mar  = ['o', 's', '*', 'D','H']
colors = ['red', 'blue', 'black', 'green', 'orange', 'purple']

try:
    import medcoupling as mc  # Replace 'some_module' with the name of the module you want to load
    print("medcoupling loaded successfully!")
except ImportError as e:
    print("Failed to load module:", e)
    try:
        import scienceplots
        plt.style.use('science')
    except ImportError as e:
        print("Failed to load module:", e)
import os
current_directory = os.getcwd() 
from scipy import special
from scipy.optimize import fsolve
from scipy.integrate import quad
from scipy.special import erf

In [ ]:
Cp_v = 2.077e3   # Specific heat capacity of vapor (J/kg·K)
lambda_v = 0.025  # Thermal conductivity of vapor (W/m·K)
h_lg = 2.256e6

rho_l = 958.37
Cp_l=4.21e3
rho_v = 0.597
sigma = 5.89e-2
lambda_l = 0.679  

In [ ]:
def integrand(y, beta):
    return np.exp(-(beta**2) * ((1-y)**(-2) - 2*(1 - rho_v / rho_l)*y - 1))
def equation(beta, dT):
    integral, _ = quad(integrand, 0, 1, args=(beta))
    Ja= rho_l*Cp_l*dT/(rho_v*(h_lg+(Cp_l-Cp_v)*dT))
    return 2 * (beta**2) * integral- Ja
def R_position(t, beta,tshift):
    return 2.0 * beta * np.sqrt(lambda_l / (rho_l * Cp_l) * (t+tshift))
    
# R interface positon 
def T_l_expr1(r, beta, R=0.05e-3):
    integral, _ = quad(integrand, 1 - R / r, 1, args=(beta))
    term1 = (2 * beta**2) * dT
    term2 = (rho_v * h_lg) / (rho_l * Cp_l*dT)
    return dT - term1 * term2 * integral
    
from math import sqrt, pi, floor, log10
def dt_popinet(dx):
    rho_m = (rho_v+rho_l)/2.
    n_sig_figs = 2
    dt = sqrt((rho_m/pi/sigma)*(dx)**3)
    scale = -int(floor(log10(abs(dt))) - (n_sig_figs - 1))
    return round(dt, scale)

def dt_diffusion(dx):
    alpha_l = lambda_l / (rho_l * Cp_l)
    n_sig_figs = 2
    dt = 0.5*(dx)**2/alpha_l
    scale = -int(floor(log10(abs(dt))) - (n_sig_figs - 1))
    return round(dt, scale)
    
# conditions_initiales { Temperature_thermique Champ_Don_lu dom_solide 1 examplebis.rep }
def generate_temp(L,N, beta, R0, directory):
    gridx = np.linspace(0, L, N)
    gridy = np.linspace(0, L, N)
    xc = (gridx[1:] + gridx[:-1])/2.
    yc = (gridy[1:] + gridy[:-1])/2.
    
    X, Y = np.meshgrid(xc,yc)
    mypx = X.ravel()  # using ravel
    mypy = Y.ravel()  # using flatten
    elem = mypx.size
   
    path = os.path.join(run.BUILD_DIRECTORY, directory)
    with open(os.path.join(path, "examplebis.rep"), "w") as file:
        file.write(str(elem) + "\n")
        for ielem in range(elem):
            # myval = funciton (mypx, mypy)
            myval = T_l_expr1(sqrt(mypx[ielem]**2+mypy[ielem]**2), beta, R0)
            formatted_number = f"{mypx[ielem]:.6g} {mypy[ielem]:.6g} {myval:.6g}\n"
            file.write(formatted_number)

# Configuration for simu

## dT and beta

In [ ]:
dT = 1.0

# resolution equation 
beta_guess = 1 
beta_solution = fsolve(equation, beta_guess, args=(dT))
print(f'Residus: {equation(beta_solution, dT)}') # verification de convergence en 0
print(f"The value of beta is: {beta_solution[0]}")

## Initial radius *R0*, domaine length *L* and maximum simulaiton time *tmax*

In [ ]:
beta = beta_solution[0]
R0 = 1e-3
t0 = (R0/2./beta)**2*rho_l*Cp_l/lambda_l

# By default, ONLY run for a very short time [t0, 2t0]
# Change to True for long time run [t0, 4t0]
RUN_for_4t = False   

# by default, auto-run using scirpt, True will cause pb
# the domaine is not cut in the right way. e.g., 9x1 NOT 3x3
# change this to True when running seperating validation case in terminal
# Run_file -parallel_run 
RUN_terminal = False


if RUN_for_4t:
    tmax = 3*t0
    Rf = R_position(tmax,beta,t0)
    L = 4.*R0
    nombre_de_noeuds = [41, 81, 161]
else:
    tmax = t0
    Rf = R_position(tmax,beta,t0)
    L = Rf*1.5
    nombre_de_noeuds = [41, 65, 81]

        
dt_max = [dt_popinet(L/(N-1))*0.05 for N in nombre_de_noeuds ]

prox = [1, 2, 3]
proy = [1, 2, 3]

shift_secmem2 = ['shift_secmem2', ' ', ' ']
new_source = ['new_mass_source', 'new_mass_source', ' ']

configure = ['shift_secmem2', 'new_source', 'old']   # New source term with shift ; New source term; defalult without source term
# configure = ['shift_secmem2']
# configure = ['new_source', 'old']

In [ ]:
from trustutils import run
run.introduction("L. WEI","26/11/2024")
# Declaration of the TRUST version
run.TRUST_parameters("1.9.3")

run.reset()

for x in range(len(configure)):
    for y in range(len(nombre_de_noeuds)) :
    # for y in range(1) :
        fname = configure[x]+'/maillage_'+str(nombre_de_noeuds[y]-1)
        name = 'maillage_'+str(nombre_de_noeuds[y]-1)
        substitutions_dict = {"new_mass_source" : new_source[x],
                              "shift_secmem2" : shift_secmem2[x],
                              "L" : f'{L:.4g}',
                              "R" : f'{R0}',
                              "nbn" : str(nombre_de_noeuds[y]),
                              "mtmax" : str(tmax),
                              "prox" : str(prox[y]),
                              "proy" : str(proy[y]),
                              "dT" : str(dT),
                              "mdtpost" : f'{tmax/20.:.3g}',
                              "dtprob" : f'{tmax/200.:.3g}',
                              "nbx" : str(int(2.*nombre_de_noeuds[y])),
                              "mdtmax" :str(dt_max[y])
                              }

        tc = run.addCaseFromTemplate("scriven.data"
                                 ,targetDirectory=f"{fname}"
                                 ,dic=substitutions_dict
                                 ,nbProcs=prox[y]*proy[y]
                                 ,targetData=f"{name}.data"
                                 )
    
        generate_temp(L, N = nombre_de_noeuds[y], beta=beta, R0=R0, directory=fname)
        
        
        if ((prox[y]*proy[y] > 1)):
            # partition by hand, 3x3 ou 2x2
            if RUN_terminal:
                os.system(f"cd {current_directory}; cd build/{fname};make_PAR.data {name}.data; mv PAR_{name}.data {name}.data; cd {current_directory};" )
            else:
                tc.partition()
            
run.printCases()


In [ ]:
# Run all cases
run.runCases()

In [ ]:
## Performances calculs

run.tablePerf()

## Initial temperature along r

Placed here because we cannot have plots before ``` run.runCases() ```. Otherwise test cases will show the plots when running... ```plt.show()``` is the problem actually)

In [ ]:
r_ana = np.linspace(0.75*R0, L, 500)
Tl_ana = np.asarray([T_l_expr1(r, beta, R=R0) for r in r_ana])
Tl_final = np.asarray([T_l_expr1(r, beta, R=Rf) for r in r_ana])

plt.plot(r_ana/R0, Tl_ana/dT, label=r'$T_l(r)$')
plt.plot(r_ana/R0, Tl_final/dT, label=r'$T_l(r)$')

plt.xlabel(r'$r/R_0$')
plt.ylabel(r"$(T_l-T_\mathrm{sat})/\Delta T_{\infty}$")   
# plt.title(f'Temperature $T_l$ at t_0 = {t0*1.e3:.2g}ms')
# plt.legend()
plt.ylim([0, 1.05])
plt.show()

# Results and Postprocessing

In [ ]:
# Pre-load module
def get_prb_data(file, time):
    from trustutils.files import SonSEGFile
    import numpy as np
    son_file = f'{file}'
    donne = SonSEGFile(son_file,None)
    compo = 0
    ncompo = donne.getnCompo()
    entries = donne.getEntries()
    
    # y_label = entries[compo].split()[0]
    # x_label = donne.getXLabel()
    # print("x_label : ", x_label)
    # print("y_label : ", y_label)
    # start, end = donne.getXTremePoints()
    # print("dom xmin and ymin : ", start)
    # print("dom xmax and ymax : ", end)
    
    t = donne.getValues(entries[0])[0]
    ## find closest value of t
    if time == None:
        idx = -1
    else:
        idx = (np.abs(t - time)).argmin()
        
    X = donne.getXAxis()  # length along the line p1p2, origin at p1
    Y = []
    for i in entries[compo::ncompo]:
        Y.append(list(donne.getValues(i)[1])[idx])
    if X[0] != X.min():
        X = X[::-1]
        Y = Y[::-1]
    return X, np.asarray(Y), t[idx]


def get_position_bis(fname, name, biaxi=False):
    import numpy as np
    import os, math
    if not os.path.exists(f"build/{fname}/vap_vol.txt"):
        os.system(f'grep "^Volume_phase_0" build/{fname}/{name}.err | awk \'{{print $4, $2}}\' > build/{fname}/vap_vol.txt')

    data = np.loadtxt(f'build/{fname}/vap_vol.txt')
    time = data[:,0]
    vol = data[:, 1] 
    if biaxi:
        posi = (np.array(vol)*3./2./math.pi)**(1./3.)
    else:
        posi = (np.array(vol)*4./math.pi)**(1./2.)
    return time, posi

## Temperature profil along r at different angle ( t = 2t0 )

In [ ]:
position = ['ABSCISSE', 'ORDONNEE', 'ANGLE']

for itemC in configure:
    num_mar = 0
    
    y = 2
    fname = itemC+'/maillage_'+str(nombre_de_noeuds[y]-1)
    if RUN_terminal:
        name = 'maillage_'+str(nombre_de_noeuds[y]-1)
    else:
        name = ((prox[y]*proy[y] > 1)) * "PAR_" + 'maillage_'+str(nombre_de_noeuds[y]-1)
    for i in [0, 1, 2]:
        X, T, t_loc = get_prb_data(f'build/{fname}/{name}_T_{position[i]}.son', tmax)
        # print(t_loc/t0)
        
        # freq = int(len(X)/20)
        freq = 1
        plt.scatter(X[::freq]/R0,T[::freq]/dT
                 ,marker=Mar[num_mar%len(Mar)], s=30, facecolors='none', edgecolors=colors[(num_mar)%len(colors)]
                 # ,  markevery= 60
             ,  label=f'{position[i]}'
            )
        num_mar = num_mar +1
 
    r_ana = np.linspace(0.5*R0, L, 101)
    Tl_ana = np.asarray([T_l_expr1(r, beta, R_position(t_loc,beta,t0)) for r in r_ana])
    
    plt.plot(r_ana/R0,Tl_ana/dT
             , color=colors[(num_mar)%len(colors)]
             ,  label='Ana.'
            )
    num_mar = num_mar +1
    
    
    plt.ylim(0, 1)
    plt.xlim(1, L/R0)
    plt.xlabel(r"$\varrho/R_0$")
    plt.ylabel(r"$(T_l-T_{sat})/$"+r"$\Delta $"+r'$T_{\infty}$')
    plt.legend(loc="best")
    plt.title(f'{itemC}'+r"$\Delta x=$"+f'{L/(nombre_de_noeuds[y]-1)*1.e6:.2g}'+r'$\mu m$')
    # plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
    # plt.savefig(f'Comparison_T_ana.eps', bbox_inches='tight')
    plt.show()

## Mesh convergence for temperature

In [ ]:
position = ['ABSCISSE', 'ORDONNEE', 'ANGLE']

for itemC in configure:
    num_mar = 0
    for y in range(len(nombre_de_noeuds)) :
    # for y in [1, 2, 3]:
        fname = itemC+'/maillage_'+str(nombre_de_noeuds[y]-1)
        if RUN_terminal:
            name = 'maillage_'+str(nombre_de_noeuds[y]-1)
        else:
            name = ((prox[y]*proy[y] > 1)) * "PAR_" + 'maillage_'+str(nombre_de_noeuds[y]-1)
        
        # name = 'maillage_'+str(nombre_de_noeuds[y]-1)
        for i in [0]:
            X, T, t_loc = get_prb_data(f'build/{fname}/{name}_T_{position[i]}.son', tmax)
            
            freq = int(len(X)/200)+1
            plt.scatter(X[::freq]/R0,T[::freq]/dT
                     ,marker=Mar[num_mar%len(Mar)], s=30, facecolors='none', edgecolors=colors[(num_mar)%len(colors)]
                     # ,  markevery= 60
                 ,  label=r"$\Delta x=$"+f'{L/(nombre_de_noeuds[y]-1)*1.e6:.2g}'+r'$\mu m$'
                )
            num_mar = num_mar +1
     
    r_ana = np.linspace(0.5*R0, L, 101)
    Tl_ana = np.asarray([T_l_expr1(r, beta, R_position(t_loc,beta,t0)) for r in r_ana])
    
    plt.plot(r_ana/R0,Tl_ana/dT
             , color=colors[(num_mar)%len(colors)]
             ,  label='Ana.'
            )
    num_mar = num_mar +1


    plt.ylim(0, 1.05)
    plt.xlim(1, L/R0)
    plt.xlabel(r"$\varrho/R_0$")
    plt.ylabel(r"$(T_l-T_{sat})/$"+r"$\Delta $"+r'$T_{\infty}$')
    plt.legend(loc="best")
    plt.title(itemC)
    # plt.title(r"$\frac{r}{R_0}$",r"$\frac{T_l-T_{sat}}{\Delta T_{\infty}}$")
    # plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
    # plt.savefig(f'Comparison_T_ana.eps', bbox_inches='tight')
    plt.show()

## Radius at different time instant

In [ ]:
t_ana = np.linspace(0, tmax, 20)
R_ana = R_position(t_ana,beta,t0)

def slice_with_last_array(data, freq):
    return np.concatenate((data[::freq], data[-1:] if data[-1] not in data[::freq] else []))

for itemC in configure:
    num_mar = 0
    plt.plot(t_ana/t0+1, R_ana/R0
         # ,marker=Mar[num_mar%len(Mar)], s=30, facecolors='none', edgecolors=colors[(num_mar)%len(colors)]
         , color=colors[(num_mar)%len(colors)]
         ,  label='Eq. (B.1)'
        )
    num_mar = num_mar +1
    
    for y in range(len(nombre_de_noeuds)) :
        fname = itemC+'/maillage_'+str(nombre_de_noeuds[y]-1)
        if RUN_terminal:
            name = 'maillage_'+str(nombre_de_noeuds[y]-1)
        else:
            name = ((prox[y]*proy[y] > 1)) * "PAR_" + 'maillage_'+str(nombre_de_noeuds[y]-1)
        
        t_loc, r_loc = get_position_bis(fname, name, True)
      
            
        freq = int(len(t_loc)/30)+1
        plt.scatter( slice_with_last_array(t_loc, freq)/t0+1, slice_with_last_array(r_loc, freq)/R0
                ,marker=Mar[num_mar%len(Mar)], s=20, facecolors='none', edgecolors=colors[(num_mar)%len(colors)]
                 # ,  markevery= 60
                 , color=colors[(num_mar)%len(colors)]
             ,  label=f'{L/(nombre_de_noeuds[y]-1)*1.e6:.2g}'+r'$\mu m$'
            )
        num_mar = num_mar +1
        
        
    
    # plt.xlim([1,1.4])
    # plt.ylim([1,1.2])
    
    plt.xlabel(r"$t/t_0$")
    plt.ylabel(r"$R/R_0$")
    plt.legend(loc="best")
    plt.title(itemC)
    # plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
    # plt.savefig(f'build/CVG_R.eps', bbox_inches='tight')
    plt.show()

## Error of interface position at the end of simulation 

In [ ]:
num_mar = 0
for itemC in configure:
    err = []
    xxx = []
    for y in range(len(nombre_de_noeuds)) :
        fname = itemC+'/maillage_'+str(nombre_de_noeuds[y]-1)
        # name = 'maillage_'+str(nombre_de_noeuds[y]-1)
        name = (((prox[y]*proy[y] > 1)) * "PAR_") + 'maillage_'+str(nombre_de_noeuds[y]-1)
        
        t_loc, r_loc = get_position_bis(fname, name, True)
        x_simu = r_loc[-1]
        x_theric = R_position(t_loc[-1],beta,t0)
        err.append(abs(x_simu-x_theric)/x_theric)
        xxx.append(nombre_de_noeuds[y]-1)
    
    plt.scatter(xxx, err, label = itemC
               ,marker=Mar[num_mar%len(Mar)], s=20, facecolors='none', edgecolors=colors[(num_mar)%len(colors)])
    num_mar = num_mar + 1
    
x = np.asarray(xxx)
# plt.plot(x, 1/x**2, label='2nd order')  
plt.plot(x, err[0]*xxx[0]/x, label='1st order')

plt.xlabel(r"Number of cells")
plt.ylabel(r"Relative error")    
plt.xscale('log', base=2)
plt.yscale('log')
plt.legend(loc="best")
# plt.title(r"$\frac{r}{R_0}$",r"$\frac{T_l-T_{sat}}{\Delta T_{\infty}}$")
# plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
plt.savefig(f'build/error.eps', bbox_inches='tight')
plt.show()

## Interface shape and interface velocity at the end of simulation 

In [ ]:
from trustutils import visit
colors_line = [(255, 0, 0, 255), (0, 255, 0, 255), (0, 0, 255, 255)]

for index, itemC in enumerate(configure):
    y = -1 # finest mesh
    fname = itemC+'/maillage_'+str(nombre_de_noeuds[y]-1)
    flata = f'{fname}/lata/post.lata'
    
    visu = visit.Show(flata,"Mesh","INTERFACES")
    visu.visuOptions(["no_databaseinfo"])  
    visu.visitCommand(f'SetActivePlots(({index}))')
    visu.visitCommand(f'm{index}=MeshAttributes()')
    visu.visitCommand(f'm{index}.opaqueMode=m{index}.Off')
    visu.visitCommand(f'm{index}.foregroundFlag=0')
    visu.visitCommand(f'm{index}.meshColor={colors_line[index]}')
    visu.visitCommand(f'm{index}.lineWidth = 2')
    visu.visitCommand(f'SetPlotOptions(m{index})')
    visu.visitCommand(f'm{index}.legendFlag = 0')
    visu.visitCommand(f'SetPlotOptions(m{index})')

    
    visu.addField(flata,"Vector","VITESSE_SOM_INTERFACES")
    index_v = len(configure) + index
    visu.visitCommand(f'SetActivePlots(({index_v}))')
    visu.visitCommand(f'm{index_v}=VectorAttributes()')
    visu.visitCommand(f'm{index_v}.useLegend = 0')
    visu.visitCommand(f'm{index_v}.colorByMag = 0')
    visu.visitCommand(f'm{index_v}.useStride = 1')
    visu.visitCommand(f'm{index_v}.vectorColor = {colors_line[index]}')
    visu.visitCommand(f'SetPlotOptions(m{index_v})')

    visu.visitCommand("V=View2DAttributes()")
    visu.visitCommand(f"V.windowCoords = (0, {L}, 0, {L})")
    visu.visitCommand("V.viewportCoords = (0.15, 0.95, 0.1, 0.95) ")
    visu.visitCommand("SetView2D(V)")

    visu.visitCommand('title = CreateAnnotationObject("Text2D")')
    visu.visitCommand(f'title.text = "{itemC}"')
    visu.visitCommand(f'title.position = (0.45, 0.9)')
    visu.visitCommand(f'title.fontBold = 1')

    visu.plot() 

In [ ]:
from trustutils import visit
colors_line = [(255, 0, 0, 255), (0, 255, 0, 255), (0, 0, 255, 255)]

for index, itemC in enumerate(configure):
    y = -1 # finest mesh
    fname = itemC+'/maillage_'+str(nombre_de_noeuds[y]-1)
    flata = f'{fname}/lata/post.lata'
    if index == 0:
        visu = visit.Show(flata,"Mesh","INTERFACES")
        visu.visuOptions(["no_databaseinfo"])
    else:
        visu.addField(flata,"Mesh","INTERFACES")
        visu.visitCommand(f'SetActivePlots(({index}))')
        visu.visitCommand(f'm{index}=MeshAttributes()')
        visu.visitCommand(f'm{index}.opaqueMode=m{index}.Off')
        visu.visitCommand(f'm{index}.foregroundFlag=0')
        visu.visitCommand(f'm{index}.meshColor={colors_line[index]}')
        visu.visitCommand(f'm{index}.lineWidth = 2')
        visu.visitCommand(f'SetPlotOptions(m{index})')
        visu.visitCommand(f'm{index}.legendFlag = 0')
        visu.visitCommand(f'SetPlotOptions(m{index})')
        
visu.visitCommand("V=View2DAttributes()")
visu.visitCommand(f"V.windowCoords = (0, {L}, 0, {L})")
visu.visitCommand("V.viewportCoords = (0.15, 0.95, 0.1, 0.95) ")
visu.visitCommand("SetView2D(V)")
visu.plot() 